In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "4"   

In [3]:
import numpy as np
import pandas as pd
import spacy
from difflib import SequenceMatcher
from sklearn.metrics import precision_recall_fscore_support
from sacrebleu.metrics import TER
import torch
import numpy as np
from captum.attr import LayerIntegratedGradients

/data/miniconda3/envs/hlt/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
import sys
sys.path.append("/code/HLTproject_code/task_1B_complexity_explanation/baselines")

from explanations import evaluate, silver_labels

In [4]:
# test = pd.read_parquet("/code/HLTproject_code/data/test.parquet")

# orig = (test[test["is_original"]]
#         .drop_duplicates("original_sentence_idx")
#         .set_index("original_sentence_idx")["text"])
# simp = (test[~test["is_original"]]
#         .drop_duplicates("original_sentence_idx")   # 1 semplificazione per originale (K=1)
#         .set_index("original_sentence_idx")["text"])

# pairs = (pd.concat([orig.rename("original"), simp.rename("simplification")], axis=1)
#            .dropna().reset_index())
# print(f"Coppie di valutazione (frasi complesse del test): {len(pairs)}")
# pairs.head()

In [5]:
#nlp = spacy.load("it_core_news_md")

# eval_set = []
# for _, row in pairs.iterrows():
#     doc_o = [t for t in nlp(row["original"]) if not t.is_space]
#     tokens, labels, tipo = silver_labels(row["original"], row["simplification"])
#     eval_set.append({
#         "idx": row["original_sentence_idx"],
#         "tokens": tokens,
#         "lemmas": [t.lemma_.lower() for t in doc_o],
#         "pos":    [t.pos_ for t in doc_o],
#         "silver": labels, "tipo": tipo,
#         "simp_tokens": [t.text for t in nlp(row["simplification"]) if not t.is_space],
#     })

In [5]:
import pickle

# eval_set salvato a monte 
with open("/code/HLTproject_code/task_1B_complexity_explanation/data/eval_set_full.pkl", "rb") as f:
    eval_set = pickle.load(f)

In [6]:
def predict_ig(eval_set, model, tokenizer, top_frac=0.34, n_steps=50, device="cuda"):
    """
    Integrated Gradients sul BERT (classificazione o regressione).
    Ritorna preds binari allineati a eval_set[i]['tokens'].
    """
    model.eval().to(device)
    emb_layer = model.get_input_embeddings()

    def forward_fn(input_ids, attention_mask):
        logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
        # classificazione -> logit della classe "complesso" (1); regressione -> unico output
        return logits[:, 1] if logits.shape[-1] > 1 else logits.squeeze(-1)

    lig = LayerIntegratedGradients(forward_fn, emb_layer)
    preds = []

    for ex in eval_set:
        # (2) tokenizzo i token spaCy gia' splittati -> word_ids() mappa subword -> token spaCy
        enc = tokenizer(ex["tokens"], is_split_into_words=True, return_tensors="pt",
                        truncation=True, max_length=128)
        input_ids      = enc["input_ids"].to(device)
        attention_mask = enc["attention_mask"].to(device)
        word_ids       = enc.word_ids(0)              # richiede tokenizer "fast"

        # baseline: PAD al posto dei token reali, mantenendo i token speciali
        ref = input_ids.clone()
        special = tokenizer.get_special_tokens_mask(input_ids[0].tolist(),
                                                    already_has_special_tokens=True)
        for k, is_sp in enumerate(special):
            if not is_sp:
                ref[0, k] = tokenizer.pad_token_id

        # (1) attribuzioni IG sull'embedding, sommate sulla dimensione nascosta
        attr = lig.attribute(inputs=input_ids, baselines=ref,
                             additional_forward_args=(attention_mask,), n_steps=n_steps)
        attr = attr.sum(dim=-1).squeeze(0).detach().cpu()      # [seq_len]

        # (2) aggrego i subword sul token spaCy
        n = len(ex["tokens"])
        agg = np.zeros(n)
        for k, wid in enumerate(word_ids):
            if wid is not None and wid < n:
                agg[wid] += float(attr[k])

        # (3) soglia: evidenzio il top_frac dei token piu' "responsabili"
        k_top = max(1, int(round(top_frac * n)))
        top   = np.argsort(agg)[-k_top:]
        p = [0] * n
        for i in top:
            p[i] = 1
        preds.append(p)

    return preds

In [ ]:
# from transformers import AutoModelForSequenceClassification, AutoTokenizer

# ckpt  = "/code/HLTproject_code/task_1A_complexity_prediction/bert_reg/lr_5e-05/checkpoint-1886"   # checkpoint di regressione
# model = AutoModelForSequenceClassification.from_pretrained(ckpt)
# tok   = AutoTokenizer.from_pretrained(ckpt)

# preds_ig = predict_ig(eval_set, model, tok, top_frac=0.5, device="cuda")
# print("BERT+IG:", evaluate(eval_set, preds_ig))

In [ ]:
## valori vecchi quando la soglia era top_frac=0.5, ora cambiata a top_frac=0.34 

# BERT+IG: {'precision': 0.37645224777156006, 'recall': 0.5546107334056061, 'f1': 0.40910565335363935, 
# 'ed_sub1': 19.637883008356546, 'ed_sub1.5': 22.28544899854092, 
# 'ed_sub2': 24.341424592120973, 'ter': 77.19913905976591, 'n_valutate': 7539}

## risultati con top_frac=0.34

#BERT+IG: {'precision': 0.39156390109886646, 'recall': 0.404027722294076, 
# 'f1': 0.3577473737765195, 'ed_sub1': 19.371402042711235, 'ed_sub1.5': 22.188884467436, 'ed_sub2': 24.447008887120308, 'ter': 79.07507699850679, 'n_valutate': 7539}

In [7]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

ckpt = "/code/HLTproject_code/task_1A_complexity_prediction/bert_cls/lr_5e-05/checkpoint-2828"  # checkpoint di classificazione
model = AutoModelForSequenceClassification.from_pretrained(ckpt)
tok   = AutoTokenizer.from_pretrained(ckpt)

preds_ig = predict_ig(eval_set, model, tok, top_frac=0.34, device="cuda")
print("BERT+IG:", evaluate(eval_set, preds_ig))

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 9128.16it/s]


OK: tutte 7539 le frasi sono allineate (tokens == silver == pred).
BERT+IG: {'precision': 0.39156390109886646, 'recall': 0.404027722294076, 'f1': 0.3577473737765195, 'ed_sub1': 19.371402042711235, 'ed_sub1.5': 22.188884467436, 'ed_sub2': 24.447008887120308, 'ter': 79.07507699850679, 'n_valutate': 7539}


In [8]:
import pickle
with open('/code/HLTproject_code/task_1B_complexity_explanation/preds/preds_ig.pkl', 'wb') as f:
    pickle.dump(preds_ig, f)